### Top-$1$ and Top-$k$ Accuracy Test for Classification Models
This notebook performs top-$1$ and top-$k$ accuracy test for classification models that are available at DeGirum model zoo.<br>
* You need to specify your cloud API access token and cloud zoo URL in [env.ini](env.ini)<br>
* You also need to specify the following in the code below: 
  * `MODEL_STR` &mdash; a model identifier string
  * `IMAGES_DIR` &mdash; path to folder containing test images
  * `JSON_FILENAME` &mdash; name of JSON file describing the labels of test images
  * `NUM_CLASSES` &mdash; number of classes in the test dataset
  * `K` &mdash; $k$ in top-$k$<br>
* You can optionally specify the following in the code below:
  * `VERBOSE` &mdash; set print verbosity

#### Specify test options here

In [ ]:
# Specify a classification model from DeGirum model zoo
MODEL_STR = "mobilenet_v2_imagenet--224x224_float_n2x_cpu_1"

# Specify path to the folder containing test images and JSON file
IMAGES_DIR = "../val_images"

# Specify name of the JSON file describing the labels of test images
# Note: JSON file should contain {<img1> : <label1>, <img2> : <label2>, ...} map and be placed inside IMAGES_DIR above
JSON_FILENAME = "../val_images/image_label_map.json"

# Specify the number of classes in the test dataset 
# For e.g., MNIST and FashionMNIST have 10 classes, ImageNet has 1000 classes
NUM_CLASSES = 1000

# Specify the value of `k` in top-k
K = 5

#### Specify where do you want to run your inferences

In [ ]:
import csv
import degirum as dg,mytools
import json
import os
from classification_accuracy_utils import *
# Get cloud API access token from env.ini file
CLOUD_API_TOKEN = mytools.get_token()

# Get cloud zoo URL from env.ini file
CLOUD_ZOO_URL = mytools.get_cloud_zoo_url()


# Please UNCOMMENT only ONE of the following lines to specify where to run AI inference

# 1. Inference on the DeGirum Cloud Platform
zoo = dg.connect(dg.CLOUD, CLOUD_ZOO_URL, CLOUD_API_TOKEN)

# 2. Inference on DeGirum AI Server deployed on a localhost or on some computer in your LAN or VPN
# zoo = dg.connect(mytools.get_ai_server_hostname(), CLOUD_ZOO_URL, CLOUD_API_TOKEN)

# 3. Inference on DeGirum ORCA accelerator installed on your computer
# zoo = dg.connect(dg.LOCAL, CLOUD_ZOO_URL, CLOUD_API_TOKEN)


### Load the evaluation model

In [ ]:
model = zoo.load_model(MODEL_STR)

#### Computing Top-k accuracy using input parameters

In [ ]:
top_k_accuracy = mytools.ImageClassificationModelEvaluator(model,IMAGES_DIR,JSON_FILENAME,NUM_CLASSES,K)
results = top_k_accuracy.evaluate()

#### Computing Top-k accuracy using input yaml config

In [ ]:
top_k_accuracy = mytools.ImageClassificationModelEvaluator.init_from_yaml(model,"config.yaml")
results = top_k_accuracy.evaluate()

### Saving the results to a csv file

In [ ]:
if not os.path.exists('topaccuracy'):
    os.mkdir('topaccuracy')
csv_file_path = os.path.join('topaccuracy', 'topacc_class_models.csv')
if not os.path.isfile(csv_file_path):
    with open(csv_file_path, 'x') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['model', 'top-1_acc', 'top-'+str(results[0])+'_acc'])
        writer.writerow([MODEL_STR, results[1], results[2]])
else:
    with open(csv_file_path, 'a') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([MODEL_STR, results[1], results[2]])
        